# Deduplicate Targeting List - User Processing

This notebook processes user lists across multiple Excel sheets, deduplicates users, and maps OR user IDs.

In [1]:
import pandas as pd
from openpyxl import load_workbook
import random

# Check current directory
# !pwd

## Step 1: Load Excel File and Get Sheet Names

In [2]:
file_ = 'Menu_Targeting_User_Segments_202610.xlsx'
df_excel = pd.ExcelFile(file_)
sheets = []
for sheet_name in df_excel.sheet_names:
    sheets.append(sheet_name)
    
print(f'Total sheets: {len(sheets)}')
sheets

Total sheets: 6


['Sleeping User (A)',
 'Warm User',
 'New User(of previous mos)',
 'Booking Standing 5 Star',
 'Sleeping User (B)',
 'Buffet Lovers']

## Step 2: Deduplicate Users Across All Sheets

This step processes each sheet and:
- Removes duplicate users that appear in previous sheets
- Splits deduplicated users into 'out' and 'in' groups (50/50)
- Saves the results back to Excel

In [3]:
all_users_list = []
all_users_sheet_distinct_user = {}
last_sheet_user_count = 0

print(f'Total sheets to process: {len(sheets)}')
print(f'Sheets: {sheets}\n')

# Process each sheet
for idx, sheet in enumerate(sheets, 1):
    try:
        print(f'\n[{idx}/{len(sheets)}] Processing sheet: {sheet}')
        
        # Read the sheet
        df = pd.read_excel(file_, sheet_name=sheet)
        print(f'  Total users in sheet: {len(df)}')
        
        # Get user list and deduplicate
        temp_user_list = df.userid.to_list()
        
        # --- 🔹 MODIFY FROM HERE ---
        # Step 1: Deduplicate within the same sheet (preserve order)
        unique_within_sheet = list(dict.fromkeys(temp_user_list))
        print(f'  Duplicates within sheet: {len(temp_user_list) - len(unique_within_sheet)}')
        
        # Step 2: Remove users already seen in previous sheets
        current_sheet_user_list = [u for u in unique_within_sheet if u not in all_users_list]
        
        # Step 3: Ensure uniqueness (safety check)
        current_sheet_user_list = list(dict.fromkeys(current_sheet_user_list))
        # --- 🔹 MODIFY UNTIL HERE ---
        
        # Track duplicates
        dup_list = set([item for item in unique_within_sheet if item not in current_sheet_user_list])
        all_users_sheet_distinct_user[sheet] = current_sheet_user_list
        count_dict = {key: len(dup_list.intersection(set(value))) for key, value in all_users_sheet_distinct_user.items()}
        print(f'  Duplicate sources: {count_dict}')
        print(f'  Unique users (after dedup): {len(current_sheet_user_list)}')
        print(f'  Total duplicates removed: {len(temp_user_list) - len(current_sheet_user_list)}')
        
        # --- 🔹 MODIFY THIS LINE TOO ---
        # Update global user list *after* deduplication
        all_users_list.extend(current_sheet_user_list)
        all_users_list = list(set(all_users_list))
        print(f'  Total unique users so far: {len(all_users_list)}')
        # --- 🔹 END MODIFICATION ---
        
        # Add deduplication column with unique values only
        df['after_dedup'] = current_sheet_user_list + (len(df) - len(current_sheet_user_list)) * ['na']
        
        # Shuffle and split for out/in columns
        random.shuffle(current_sheet_user_list)
        split_index = int(len(current_sheet_user_list) * 0.5)
        
        part1 = current_sheet_user_list[:split_index]
        part2 = current_sheet_user_list[split_index:]
        
        part1 = part1 + (len(df) - len(part1)) * ['na']
        part2 = part2 + (len(df) - len(part2)) * ['na']
        
        df['out'] = part1
        df['in'] = part2
        
        # Replace 'na' with empty strings
        df.replace('na', '', inplace=True)
        
        # Write back to Excel
        with pd.ExcelWriter(file_, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            df.to_excel(writer, sheet_name=sheet, index=False)
        
        # Add metadata to sheet
        workbook = load_workbook(file_)
        sheet_obj = workbook[sheet]
        sheet_obj['F1'] = 'deduplicate count'
        sheet_obj['F2'] = str(count_dict)
        workbook.save(file_)
        
        last_sheet_user_count = len(all_users_list)
        print(f'  ✓ Sheet saved successfully')
        
    except Exception as e:
        print(f'  ✗ ERROR processing sheet "{sheet}": {e}')
        import traceback
        traceback.print_exc()
        break

print(f'\n=== DEDUPLICATION COMPLETE ===')
print(f'Processed {idx} of {len(sheets)} sheets')
print(f'Total unique users: {last_sheet_user_count}')

Total sheets to process: 6
Sheets: ['Sleeping User (A)', 'Warm User', 'New User(of previous mos)', 'Booking Standing 5 Star', 'Sleeping User (B)', 'Buffet Lovers']


[1/6] Processing sheet: Sleeping User (A)
  Total users in sheet: 45677
  Duplicates within sheet: 0
  Duplicate sources: {'Sleeping User (A)': 0}
  Unique users (after dedup): 45677
  Total duplicates removed: 0
  Total unique users so far: 45677
  ✓ Sheet saved successfully

[2/6] Processing sheet: Warm User
  Total users in sheet: 11075
  Duplicates within sheet: 0
  Duplicate sources: {'Sleeping User (A)': 0, 'Warm User': 0}
  Unique users (after dedup): 11075
  Total duplicates removed: 0
  Total unique users so far: 56752
  ✓ Sheet saved successfully

[3/6] Processing sheet: New User(of previous mos)
  Total users in sheet: 6989
  Duplicates within sheet: 0
  Duplicate sources: {'Sleeping User (A)': 0, 'Warm User': 0, 'New User(of previous mos)': 0}
  Unique users (after dedup): 6989
  Total duplicates removed: 0
  T

## Step 3: Map OR User IDs

Query the database to get the mapping between Mars user IDs and OpenRice user IDs.

In [4]:
import pymssql

DB_info = {
    'server': '192.168.61.119:7622',
    'database': 'openrice3',
    'user': 'BAReporting',
    'password': 'KeHeCReme8he'
}

def get_sql_data(sql, columns_name=None):
    with pymssql.connect(DB_info['server'], DB_info['user'], DB_info['password'], 'mars') as conn:
        cursor = conn.cursor()
        cursor.execute(sql)
        data = cursor.fetchall()
        
    return pd.DataFrame(data) if columns_name == None else pd.DataFrame(data, columns=columns_name)

or_df = get_sql_data("""
select ou.username, ou.userid as OR_userid, mu.userid as Mars_userid, ou.ssouserid, mu.status, ou.status 
from openrice3.dbo.[user] (nolock) ou
left join mars.dbo.[user] mu (nolock) on mu.ssouserid = ou.ssouserid
where mu.userid is not null and mu.status not in (0, 5) and ou.status not in (0, 5)
""", columns_name=['username', 'oruserid', 'marsuserid','ssouserid', 'marsstatus', 'orstatus'])

or_df = or_df[['oruserid', 'marsuserid', 'marsstatus', 'orstatus']]
print(f'Loaded {len(or_df)} user mappings')
or_df.head()

Loaded 7319755 user mappings


,oruserid,marsuserid,marsstatus,orstatus
0,67853659,4705584,4,4
1,67822251,4674495,4,4
2,67626778,4489158,4,4
3,66516524,3405542,4,4
4,66527540,3415088,4,4


## Step 4: Merge OR User IDs to Sheets

Add OR user IDs to the 'out' and 'in' columns for all sheets.
**Important:** This keeps ALL original rows and only adds OR user IDs where mappings exist.

In [5]:
# Process all sheets (or specify which ones to process)
sheets_to_process = sheets  # Process all sheets, or use: ['NEW NEW User (B)']

for sheet in sheets_to_process:
    print(f'\nProcessing sheet: {sheet}')
    
    try:
        df = pd.read_excel(file_, sheet_name=sheet)
        original_row_count = len(df)
        print(f'  Original rows: {original_row_count}')
        
        # Create a clean mapping dataframe with only active users
        or_df_clean = or_df[
            (~or_df['marsstatus'].isin([0, 5])) & 
            (~or_df['orstatus'].isin([0, 5]))
        ][['marsuserid', 'oruserid']].copy()
        
        # Merge 'out' column with OR user mapping (keep all original rows)
        result_df = pd.merge(df, or_df_clean, how='left', left_on='out', right_on='marsuserid', suffixes=('', '_out'))
        result_df = result_df.drop(['marsuserid'], axis=1, errors='ignore')
        result_df = result_df.rename(columns={'out': 'out_mars_userid', 'oruserid': 'out_or_userid'})
        
        # Merge 'in' column with OR user mapping (keep all original rows)
        result_df = pd.merge(result_df, or_df_clean, how='left', left_on='in', right_on='marsuserid', suffixes=('', '_in'))
        result_df = result_df.drop(['marsuserid'], axis=1, errors='ignore')
        result_df = result_df.rename(columns={'in': 'in_mars_userid', 'oruserid': 'in_or_userid'})
        
        # Verify we didn't lose rows
        if len(result_df) != original_row_count:
            print(f'  ⚠ WARNING: Row count changed from {original_row_count} to {len(result_df)}')
        
        # Reorder columns
        base_cols = ['userid', 'after_dedup', 'out_mars_userid', 'out_or_userid', 'in_mars_userid', 'in_or_userid']
        other_cols = [col for col in result_df.columns if col not in base_cols]
        result_df = result_df[base_cols + other_cols]
        
        # Count successful mappings
        out_mapped = result_df['out_or_userid'].notna().sum()
        in_mapped = result_df['in_or_userid'].notna().sum()
        print(f'  Out mappings: {out_mapped}/{result_df["out_mars_userid"].notna().sum()}')
        print(f'  In mappings: {in_mapped}/{result_df["in_mars_userid"].notna().sum()}')
        
        # Write back to Excel
        with pd.ExcelWriter(file_, engine='openpyxl', mode='a', if_sheet_exists='overlay') as writer:
            result_df.to_excel(writer, sheet_name=sheet, index=False)
        
        print(f'  ✓ Saved {len(result_df)} rows')
        
    except Exception as e:
        print(f'  ✗ ERROR: {e}')
        import traceback
        traceback.print_exc()

print('\n=== ALL PROCESSING COMPLETE ===')


Processing sheet: Sleeping User (A)
  Original rows: 45677
  Out mappings: 22838/22838
  In mappings: 22839/22839
  ✓ Saved 45677 rows

Processing sheet: Warm User
  Original rows: 11075
  Out mappings: 5537/5537
  In mappings: 5538/5538
  ✓ Saved 11075 rows

Processing sheet: New User(of previous mos)
  Original rows: 6989
  Out mappings: 3494/3494
  In mappings: 3495/3495
  ✓ Saved 6989 rows

Processing sheet: Booking Standing 5 Star
  Original rows: 13898
  Out mappings: 5865/5865
  In mappings: 5866/5866
  ✓ Saved 13898 rows

Processing sheet: Sleeping User (B)
  Original rows: 73206
  Out mappings: 35971/35971
  In mappings: 35971/35971
  ✓ Saved 73206 rows

Processing sheet: Buffet Lovers
  Original rows: 21639
  Out mappings: 3866/3866
  In mappings: 3866/3866
  ✓ Saved 21639 rows

=== ALL PROCESSING COMPLETE ===


## Step 5: Verify Results

Check one of the sheets to verify the data looks correct.

In [6]:
# Check the last sheet to verify it has all data
df_check = pd.read_excel(file_, sheet_name=sheets[-1])
print(f'Checking sheet: {sheets[-1]}')
print(f'Columns: {list(df_check.columns)}')
print(f'Total rows: {len(df_check)}')
print(f'\nFirst 10 rows:')
df_check.head(10)

Checking sheet: Buffet Lovers
Columns: ['userid', 'after_dedup', 'out_mars_userid', 'out_or_userid', 'in_mars_userid', 'in_or_userid', 'Unnamed: 4', 'deduplicate count']
Total rows: 21639

First 10 rows:


,userid,after_dedup,out_mars_userid,out_or_userid,in_mars_userid,in_or_userid,Unnamed: 4,deduplicate count
0,154,209.0,919851.0,59397999.0,755620.0,58673162.0,NaN,"{'Sleeping User (A)': 8520, 'Warm User': 2158,..."
1,160,399.0,4868157.0,68019158.0,3882461.0,67012420.0,NaN,NaN
2,209,1058.0,1438611.0,64391810.0,821020.0,271858.0,NaN,NaN
3,385,1075.0,570052.0,30001976.0,124155.0,127610.0,NaN,NaN
4,399,1201.0,7423695.0,71096701.0,7644943.0,71383858.0,NaN,NaN
5,549,1217.0,7513178.0,71219891.0,7543162.0,71263369.0,NaN,NaN
6,561,1491.0,2699091.0,169845.0,625257.0,48840.0,NaN,NaN
7,635,1640.0,3521303.0,66274211.0,3816449.0,66944390.0,NaN,NaN
8,643,1644.0,2621029.0,65630417.0,699210.0,50192386.0,NaN,NaN
9,647,1650.0,1599533.0,64452330.0,7046444.0,70636433.0,NaN,NaN
